SPOTIFY AUTH AND USER DATA JSON GENERATION OF TOP10 TRACKS 

In [1]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import json
import os
from datetime import datetime
from pathlib import Path

# Spotify API Credentials
# Get these from: https://developer.spotify.com/dashboard
CLIENT_ID = "1f9e53ad80154199be682d2b8f8fe97d"
CLIENT_SECRET = "7d0e15d4af71469492ac4e8d5f6e1e0e"
REDIRECT_URI = "http://127.0.0.1:8000/callback"


def setup_spotify_auth():
    """
    Setup Spotify OAuth authentication
    """
    try:
        sp_oauth = SpotifyOAuth(
            client_id=CLIENT_ID,
            client_secret=CLIENT_SECRET,
            redirect_uri=REDIRECT_URI,
            scope="user-top-read"
        )
        return spotipy.Spotify(auth_manager=sp_oauth)
    except Exception as e:
        print(f"❌ Authentication error: {e}")
        return None
    
import requests

LASTFM_API_KEY = "a310c4b22347dd9e733a81eb8970d7d6"

def get_lastfm_track_info(artist_name, track_name):
    """
    Helper function to fetch tags and playcount from Last.fm
    """
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "track.getInfo",
        "api_key": LASTFM_API_KEY,
        "artist": artist_name,
        "track": track_name,
        "autocorrect": 1,  # Tells Last.fm to fix minor spelling errors
        "format": "json"
    }
    
    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()
        
        # Check if Last.fm returned an error (e.g., Track not found)
        if "error" in data:
            print(f"⚠ Last.fm couldn't find data for: {artist_name} - {track_name}")
            return {"tags": [], "playcount": 0}
            
        track_info = data.get("track", {})
        playcount = int(track_info.get("playcount", 0))
        tags_data = track_info.get("toptags", {}).get("tag", [])
        
        if isinstance(tags_data, dict):
            tags_data = [tags_data]
            
        tags = [t.get("name").lower() for t in tags_data if "name" in t]
        
        return {
            "tags": tags,
            "playcount": playcount
        }
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Last.fm API request failed: {e}")
        return {"tags": [], "playcount": 0}

def decide_sarcasm_params(tags, playcount):
    """
    Allot sarcasm parameters based on Last.fm tags and global playcount.
    """
    tags_str = " ".join(tags).lower()
    
    # 1. High-Art / Classical (Requires dry, intelligent sarcasm)
    if any(g in tags_str for g in ["classical", "jazz", "orchestra", "avant-garde", "progressive"]):
        return {"s_level": 2, "s_type": "dry", "target": "situation"}
        
    # 2. Guilty Pleasures (Cheesy pop, eurodance, boy bands)
    if any(g in tags_str for g in ["eurodance", "boy band", "teen pop", "glam metal", "bro country"]):
        return {"s_level": 3, "s_type": "playful", "target": "self"}
        
    # 3. Serious / Melancholic (Safety override: disable sarcasm)
    if any(g in tags_str for g in ["sad", "funeral", "tragedy", "emo", "melancholy"]):
        return {"s_level": 0, "s_type": "none", "target": "none"}
        
    # 4. Overplayed/Mainstream Pop (Last.fm playcounts are huge, e.g., > 5 million)
    if playcount > 5000000:
        return {"s_level": 5, "s_type": "exaggerated","target": "situation"}
        
    # Default fallback
    return {"s_level": 2, "s_type": "witty", "target": "situation"}

def fetch_top_songs(sp, limit=10, time_range='medium_term'):
    """
    Fetch user's top songs from Spotify and enrich with Last.fm data
    """
    try:
        print("Fetching tracks from Spotify and enriching with Last.fm data...")
        results = sp.current_user_top_tracks(limit=limit, time_range=time_range)
        
        songs = []
        for track in results['items']:
            track_name = track['name']
            
            # Spotify returns multiple artists. We save them all for display, 
            # but ONLY use the primary artist for the Last.fm search to avoid API errors.
            all_artists = ', '.join([artist['name'] for artist in track['artists']])
            primary_artist = track['artists'][0]['name']
            album_name = track['album']['name']
            track_id = track['id']
            
            # 1. Fetch Last.fm Enrichment
            lastfm_data = get_lastfm_track_info(primary_artist, track_name)
            
            # 2. Calculate Sarcasm Parameters
            sarcasm_params = decide_sarcasm_params(lastfm_data['tags'], lastfm_data['playcount'])
            
            # 3. Build the final enriched dictionary
            song_data = {
                'track_name': track_name,
                'artist_name': all_artists,
                'album_name': album_name,
                'track_id': track_id,
                'playcount': lastfm_data['playcount'],
                'tags': lastfm_data['tags'],
                'sarcasm_level': sarcasm_params['s_level'],
                'sarcasm_type': sarcasm_params['s_type'],
                'target': sarcasm_params['target']
            }
            
            songs.append(song_data)
            
        return songs
        
    except Exception as e:
        print(f"❌ Error fetching tracks: {e}")
        return None

def save_to_json(songs, filename='top_10_songs.json'):
    """
    Save songs data to a JSON file
    
    Args:
        songs: List of song dictionaries
        filename: Output JSON filename (default: 'top_10_songs.json')
    
    Returns:
        True if successful, False otherwise
    """
    try:
        output = {
            'timestamp': datetime.now().isoformat(),
            'total_songs': len(songs),
            'songs': songs
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Successfully saved {len(songs)} songs to '{filename}'")
        return True
    except Exception as e:
        print(f"❌ Error saving to JSON: {e}")
        return False
    
def display_songs(songs):
    """
    Display songs in a formatted table, including Last.fm data and sarcasm parameters
    """
    print("\n" + "="*80)
    print("YOUR TOP 10 SPOTIFY SONGS (ENRICHED)")
    print("="*80)
    for i, song in enumerate(songs, 1):
        print(f"\n{i}. {song['track_name']}")
        print(f"   Artist: {song['artist_name']}")
        print(f"   Album: {song['album_name']}")
        
        # Display Last.fm data
        tags_display = ', '.join(song['tags']).title() if song['tags'] else 'Unknown'
        print(f"   Last.fm Tags: {tags_display}")
        print(f"   Last.fm Playcount: {song['playcount']:,}")
        
        # Display Sarcasm Mapping
        print(f"   Sarcasm Params -> Level: {song['sarcasm_level']} | Type: {song['sarcasm_type'].title()} | Target: {song['target'].title()}")
    print("\n" + "="*80)

def main():
    """
    Main function to execute the Spotify top songs fetcher
    """
    print("🎵 Spotify Top 10 Songs Fetcher\n")
    
    # Check for credentials
    if CLIENT_ID == "YOUR_CLIENT_ID" or CLIENT_SECRET == "YOUR_CLIENT_SECRET":
        print("❌ Please update CLIENT_ID and CLIENT_SECRET with your Spotify API credentials")
        print("   Get them from: https://developer.spotify.com/dashboard")
        return
    
    # Setup authentication
    print("Authenticating with Spotify...")
    sp = setup_spotify_auth()
    if not sp:
        return
    
    # Fetch songs
    print("Fetching your top 10 songs...")
    songs = fetch_top_songs(sp, limit=10, time_range='medium_term')
    
    if songs:
        # Display songs
        display_songs(songs)
        
        # Save to JSON
        save_to_json(songs)
    else:
        print("❌ Failed to fetch songs")


if __name__ == "__main__":
    main()


🎵 Spotify Top 10 Songs Fetcher

Authenticating with Spotify...
Fetching your top 10 songs...
Fetching tracks from Spotify and enriching with Last.fm data...

YOUR TOP 10 SPOTIFY SONGS (ENRICHED)

1. bargad
   Artist: sufr, Arpit Bala, toorjo dey
   Album: bargad
   Last.fm Tags: Unknown
   Last.fm Playcount: 397,762
   Sarcasm Params -> Level: 2 | Type: Witty | Target: Situation

2. Lose My Mind (feat. Doja Cat) [From F1® The Movie]
   Artist: Don Toliver, Doja Cat, F1 The Album
   Album: Lose My Mind (feat. Doja Cat) [From F1® The Movie]
   Last.fm Tags: Fernandacoxta, Soundtrack, Pop Rap, Synthpop, Electrodisco
   Last.fm Playcount: 3,574,080
   Sarcasm Params -> Level: 2 | Type: Witty | Target: Situation

3. chaandni
   Artist: sufr, Adil
   Album: amnesia: side A
   Last.fm Tags: Unknown
   Last.fm Playcount: 144,838
   Sarcasm Params -> Level: 2 | Type: Witty | Target: Situation

4. No Pole
   Artist: Don Toliver
   Album: Love Sick (Deluxe)
   Last.fm Tags: Pop, Don Toliver
   La



Fetch Lyrics from Spotify Songs
Reads songs from top_10_songs.json and fetches lyrics, saving to lyrics.json


In [3]:

import sys
import json
import os
from datetime import datetime
from lrclib import LrcLibAPI
from typing import Optional, Dict, List
from pathlib import Path

# Add current directory to path for local module imports
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = str(Path.cwd())  # Fallback for Jupyter notebooks
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

class LyricFetcher:
    """
    A utility class to fetch lyrics for tracks using LrcLib API
    """
    
    def __init__(self, user_agent: str = "spotify-lyric-fetcher/1.0.0"):
        """
        Initialize the LyricFetcher with LrcLib API
        
        Args:
            user_agent: User agent string for API requests
        """
        self.api = LrcLibAPI(user_agent=user_agent)
    
    def get_lyrics(
        self,
        track_name: str,
        artist_name: str,
        album_name: Optional[str] = None,
        duration: Optional[int] = None
    ) -> Optional[Dict[str, str]]:
        """
        Fetch lyrics for a specific track
        
        Args:
            track_name: Name of the track
            artist_name: Name of the artist
            album_name: Name of the album (optional)
            duration: Duration of the track in seconds (optional)
        
        Returns:
            Dictionary with 'synced_lyrics' and 'plain_lyrics' keys, or None if not found
        """
        try:
            lyrics = self.api.get_lyrics(
                track_name=track_name,
                artist_name=artist_name,
                album_name=album_name,
                duration=duration
            )
            
            if lyrics:
                return {
                    "synced_lyrics": lyrics.synced_lyrics,
                    "plain_lyrics": lyrics.plain_lyrics,
                    "track_name": track_name,
                    "artist_name": artist_name
                }
            return None
        except Exception as e:
            print(f"❌ Error fetching lyrics for {track_name} by {artist_name}: {e}")
            return None
    
    def get_lyrics_preview(
        self,
        track_name: str,
        artist_name: str,
        album_name: Optional[str] = None,
        duration: Optional[int] = None,
        lines: int = 5
    ) -> Optional[str]:
        """
        Fetch and return a preview of lyrics (first N lines)
        
        Args:
            track_name: Name of the track
            artist_name: Name of the artist
            album_name: Name of the album (optional)
            duration: Duration of the track in seconds (optional)
            lines: Number of lines to return (default: 5)
        
        Returns:
            Preview text of the first N lines, or None if not found
        """
        lyrics_data = self.get_lyrics(track_name, artist_name, album_name, duration)
        
        if not lyrics_data:
            return None
        
        # Prefer synced lyrics, fall back to plain lyrics
        lyrics_text = lyrics_data["synced_lyrics"] or lyrics_data["plain_lyrics"]
        
        if lyrics_text:
            preview_lines = "\n".join(lyrics_text.split("\n")[:lines])
            return preview_lines
        
        return None
    
    def fetch_lyrics_batch(self, tracks: List[Dict]) -> List[Dict]:
        """
        Fetch lyrics for multiple tracks
        
        Args:
            tracks: List of dictionaries with keys: 'track_name', 'artist_name', 'album_name' (optional), 'duration' (optional)
        
        Returns:
            List of dictionaries containing track info and lyrics
        """
        results = []
        
        for track in tracks:
            print(f"🎵 Fetching lyrics for: {track.get('track_name')} by {track.get('artist_name')}...")
            
            lyrics_data = self.get_lyrics(
                track_name=track.get("track_name"),
                artist_name=track.get("artist_name"),
                album_name=track.get("album_name"),
                duration=track.get("duration")
            )
            
            if lyrics_data:
                track_with_lyrics = {**track, **lyrics_data}
                results.append(track_with_lyrics)
                print(f"✅ Lyrics found for {track.get('track_name')}")
            else:
                print(f"⚠️  No lyrics found for {track.get('track_name')}")
        
        return results


def load_songs_from_json(filename='top_10_songs.json'):
    """
    Load songs from Spotify JSON file
    
    Args:
        filename: JSON file containing songs (default: 'top_10_songs.json')
    
    Returns:
        List of song dictionaries or None if file not found
    """
    if not os.path.exists(filename):
        print(f"❌ File not found: {filename}")
        return None
    
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        songs = data.get('songs', [])
        print(f"✅ Loaded {len(songs)} songs from {filename}")
        return songs
    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")
        return None


def fetch_all_lyrics(songs):
    """
    Fetch lyrics for all songs
    
    Args:
        songs: List of song dictionaries
    
    Returns:
        List of songs with lyrics added
    """
    fetcher = LyricFetcher()
    songs_with_lyrics = []
    
    print(f"\n🎵 Fetching lyrics for {len(songs)} songs...\n")
    
    for idx, song in enumerate(songs, 1):
        track_name = song.get('track_name')
        artist_name = song.get('artist_name')
        album_name = song.get('album_name')
        
        print(f"[{idx}/{len(songs)}] {track_name} by {artist_name}")
        
        # Fetch lyrics - try with album_name first
        lyrics_data = fetcher.get_lyrics(
            track_name=track_name,
            artist_name=artist_name,
            album_name=album_name
        )
        
        # If not found with album_name, retry without it
        if not lyrics_data:
            print(f"     ℹ️  Retrying without album name...")
            lyrics_data = fetcher.get_lyrics(
                track_name=track_name,
                artist_name=artist_name
            )
        
        # Combine song data with lyrics
        if lyrics_data:
            combined = {
                **song,
                # 'synced_lyrics': lyrics_data.get('synced_lyrics'),
                'plain_lyrics': lyrics_data.get('plain_lyrics'),
                'lyrics_found': True
            }
            print(f"     ✅ Lyrics found\n")
        else:
            combined = {
                **song,
                # 'synced_lyrics': None,
                'plain_lyrics': None,
                'lyrics_found': False
            }
            print(f"     ⚠️  No lyrics found\n")
        
        songs_with_lyrics.append(combined)
    
    return songs_with_lyrics


def save_lyrics_json(songs_with_lyrics, filename='lyrics.json'):
    """
    Save songs with lyrics to JSON file
    
    Args:
        songs_with_lyrics: List of songs with lyrics
        filename: Output JSON filename (default: 'lyrics.json')
    
    Returns:
        True if successful, False otherwise
    """
    try:
        output = {
            'timestamp': datetime.now().isoformat(),
            'total_songs': len(songs_with_lyrics),
            'songs_with_lyrics': len([s for s in songs_with_lyrics if s.get('lyrics_found')]),
            'data': songs_with_lyrics
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Successfully saved {len(songs_with_lyrics)} songs with lyrics to '{filename}'")
        return True
    except Exception as e:
        print(f"❌ Error saving to {filename}: {e}")
        return False


def display_summary(songs_with_lyrics):
    """
    Display a summary of the results
    """
    total = len(songs_with_lyrics)
    found = len([s for s in songs_with_lyrics if s.get('lyrics_found')])
    not_found = total - found
    
    print("\n" + "="*60)
    print("📊 LYRICS FETCH SUMMARY")
    print("="*60)
    print(f"Total Songs:     {total}")
    print(f"Lyrics Found:    {found} ✅")
    print(f"Lyrics Not Found: {not_found} ⚠️")
    print(f"Success Rate:    {(found/total)*100:.1f}%")
    print("="*60 + "\n")


if __name__ == "__main__":
    print("🎶 SPOTIFY LYRICS FETCHER")
    print("="*60 + "\n")
    
    # Step 1: Load songs from Spotify JSON
    songs = load_songs_from_json('top_10_songs.json')
    
    if not songs:
        print("❌ Failed to load songs. Exiting...")
        exit(1)
    
    # Step 2: Fetch lyrics for all songs
    songs_with_lyrics = fetch_all_lyrics(songs)
    
    # Step 3: Save to lyrics JSON
    success = save_lyrics_json(songs_with_lyrics, 'lyrics.json')
    
    # Step 4: Display summary
    if success:
        display_summary(songs_with_lyrics)
        print("📁 Output file: lyrics.json")
    else:
        print("❌ Failed to save lyrics JSON")


🎶 SPOTIFY LYRICS FETCHER

✅ Loaded 10 songs from top_10_songs.json

🎵 Fetching lyrics for 10 songs...

[1/10] bargad by sufr, Arpit Bala, toorjo dey
     ✅ Lyrics found

[2/10] Lose My Mind (feat. Doja Cat) [From F1® The Movie] by Don Toliver, Doja Cat, F1 The Album
     ✅ Lyrics found

[3/10] chaandni by sufr, Adil
❌ Error fetching lyrics for chaandni by sufr, Adil: 404 Not Found for https://lrclib.net/api/get?track_name=chaandni&artist_name=sufr%2C+Adil&album_name=amnesia%3A+side+A
     ℹ️  Retrying without album name...
     ✅ Lyrics found

[4/10] No Pole by Don Toliver
     ✅ Lyrics found

[5/10] Last Leaves of Autumn by Zleepyfred
     ✅ Lyrics found

[6/10] HIGHEST IN THE ROOM by Travis Scott
     ✅ Lyrics found

[7/10] After Hours by The Weeknd
     ✅ Lyrics found

[8/10] Mere Bina by Pritam, Nikhil D'Souza
     ✅ Lyrics found

[9/10] Too Many Nights (feat. Don Toliver & with Future) by Metro Boomin, Future, Don Toliver
     ✅ Lyrics found

[10/10] No Idea by Don Toliver
     ✅ 

PASSING TO EMOTION_DETECTION MODEL 

In [ ]:
import json
import torch
import os
from typing import Dict, List
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ========================= CONFIG =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# <<<=== UPDATE THIS TO YOUR ACTUAL MODEL FOLDER ===>>>
MODEL_PATH = r"C:\Users\samar\OneDrive\Desktop\prodSt\emotion_model"

EMOTION_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude",
    "grief", "joy", "love", "nervousness", "optimism", "pride",
    "realization", "relief", "remorse", "sadness", "surprise", "neutral"
]

# =========================================================

def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Model folder not found: {MODEL_PATH}")
    
    print(f"Loading model from: {MODEL_PATH}")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    
    metadata_path = os.path.join(MODEL_PATH, "metadata.json")
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    model.to(DEVICE)
    model.eval()
    print(f"✅ Model loaded | {metadata['num_labels']} emotions")
    return model, tokenizer, metadata


def predict_emotions(lyrics: str, model, tokenizer, num_labels: int, threshold: float = 0.05):
    """Return detected emotions"""
    if not lyrics or len(lyrics.strip()) < 15:
        return {
            "emotions_detected": {},
            "Emotion": ["neutral"]
        }
    
    inputs = tokenizer(
        lyrics,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )
    
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    probs = torch.sigmoid(logits).squeeze().detach().cpu().numpy()
    
    emotion_dict = {label: float(prob) for label, prob in zip(EMOTION_LABELS[:num_labels], probs)}
    
    # Get all emotions above threshold
    detected = {k: round(v, 4) for k, v in emotion_dict.items() if v >= threshold}
    detected = dict(sorted(detected.items(), key=lambda x: x[1], reverse=True))
    
    # List of detected emotion names
    emotion_list = list(detected.keys()) if detected else ["neutral"]
    
    return {
        "emotions_detected": detected,
        "Emotion": emotion_list
    }


def process_songs_file(input_file: str, output_file: str = None):
    model, tokenizer, metadata = load_model()
    
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"Processing {len(data.get('data', []))} songs...")
    
    processed = 0
    for song in data.get("data", []):
        lyrics = song.get("plain_lyrics", "")
        lyrics_found = song.get("lyrics_found", False)
        
        if lyrics_found and lyrics:
            result = predict_emotions(lyrics, model, tokenizer, metadata['num_labels'])
            song["emotion_analysis"] = {"emotions_detected": result["emotions_detected"]}
            song["Emotion"] = result["Emotion"]          # List of emotions
            processed += 1
        else:
            song["emotion_analysis"] = {"emotions_detected": {}}
            song["Emotion"] = ["neutral"]
    
    if output_file is None:
        output_file = input_file.replace(".json", "_with_emotion.json")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Processing completed!")
    print(f"   Songs processed: {processed}")
    print(f"   Output saved to: {output_file}")
    return data


# ========================= RUN =========================
if __name__ == "__main__":
    input_json_file = r"C:\Users\samar\OneDrive\Desktop\prodSt\song_data.json"   # ← Update path if needed
    
    process_songs_file(input_json_file)

GEN TEXT MODEL FOR TRACK EVAL 

In [ ]:
# import json
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from typing import List

# # ========================= CONFIG =========================
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Update this path
# GENERATION_MODEL_PATH = r"C:\Users\samar\OneDrive\Desktop\prodSt\fine_tune_model-gpt2"

# # =========================================================

# def load_generation_model():
#     print(f"Loading generation model from: {GENERATION_MODEL_PATH}")
#     tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_PATH)
#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token
    
#     model = AutoModelForCausalLM.from_pretrained(GENERATION_MODEL_PATH)
#     model.to(DEVICE)
#     model.eval()
#     print("✅ Generation model loaded")
#     return model, tokenizer


# def generate_response(
#     model, 
#     tokenizer, 
#     song: str, 
#     artist: str,
#     s_level: int = 0, 
#     s_type: str = "none", 
#     target: str = "none",
#     emotions: List[str] = None
# ) -> str:
    
#     emotion_str = ", ".join(emotions) if emotions else "neutral"
    
#     # Truncate lyrics safely

    
#     prompt = f"""<style>
# sarcasm_level: {s_level}
# sarcasm_type: {s_type}
# target: {target}
# politeness: medium
# emotions: {emotion_str}
# </style>

# <input>
# Song: {song}
# Artist: {artist}

# Emotions detected: {emotion_str}

# </input>

# <output>
# """

#     inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

#     do_sample = s_level > 0
#     temperature = 0.85 if s_level > 0 else 0.7

#     output_ids = model.generate(
#         **inputs,
#         max_new_tokens=120,
#         do_sample=do_sample,
#         temperature=temperature,
#         top_p=0.5,
#         repetition_penalty=2.0,
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )
    
#     response = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
#     return response.strip()


# def process_songs(input_json_path: str, output_json_path: str = None):
#     model, tokenizer = load_generation_model()
    
#     with open(input_json_path, 'r', encoding='utf-8') as f:
#         data = json.load(f)
    
#     print(f"Starting conversational analysis for {len(data.get('data', []))} songs...\n")
    
#     for song in data.get("data", []):
#         track_name = song.get("track_name", "Unknown")
#         artist_name = song.get("artist_name", "Unknown")
        
#         s_level = song.get("sarcasm_level", 0)
#         s_type = song.get("sarcasm_type", "none")
#         target = song.get("target", "none")
#         emotions = song.get("Emotion", ["neutral"])
        
        
#         print(f"\n{'='*80}")
#         print(f"SONG: {track_name} — {artist_name}")
#         print(f"Emotions: {', '.join(emotions)} | Sarcasm: {s_type} (Level {s_level})")
#         print(f"{'='*80}")
        
#         response = generate_response(
#             model, tokenizer, track_name, artist_name,
#             s_level, s_type, target, emotions
#         )
        
#         # Store and print immediately
#         song["generated_response"] = response
#         print(response)
#         print(f"{'='*80}\n")
    
#     # Save the updated JSON
#     if output_json_path is None:
#         output_json_path = input_json_path.replace(".json", "_with_responses.json")
    
#     with open(output_json_path, 'w', encoding='utf-8') as f:
#         json.dump(data, f, indent=2, ensure_ascii=False)
    
#     print(f"✅ All done! Responses saved to: {output_json_path}")


# # ========================= RUN =========================
# if __name__ == "__main__":
#     input_file = r"C:\Users\samar\OneDrive\Desktop\prodSt\song_data_with_emotion.json"
    
#     process_songs(input_file)

lastfm tag generated.

In [6]:
import requests
import time

API_KEY = "a310c4b22347dd9e733a81eb8970d7d6"
BASE_URL = "http://ws.audioscrobbler.com/2.0/"

tags_set = set()

def get_json(params, retries=3):
    params["api_key"] = API_KEY
    params["format"] = "json"

    for attempt in range(retries):
        try:
            res = requests.get(BASE_URL, params=params, timeout=10)

            # Debug check
            if res.status_code != 200:
                print("HTTP Error:", res.status_code)
                time.sleep(1)
                continue

            if not res.text.strip():
                print("Empty response")
                time.sleep(1)
                continue

            return res.json()

        except Exception as e:
            print("Retrying due to:", e)
            time.sleep(1)

    return {}

# Save tags
with open("lastfm_tags.txt", "w", encoding="utf-8") as f:
    for tag in sorted(tags_set):
        f.write(tag + "\n")

print(f"Collected {len(tags_set)} tags")

Collected 0 tags


Same but XLM-R (Tagger) + Mistral (Generator)

In [1]:
!pip install torch transformers accelerate bitsandbytes>=0.46.1

import json
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# ==========================================
# 1. MODEL INITIALIZATION (LOCAL LOAD)
# ==========================================

print("Loading Mistral Model from local dataset...")

# 🔥 CHANGE THIS PATH to your dataset folder
model_path = r"C:\Users\samar\OneDrive\Desktop\prodSt\mistral-4bit-local"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

mistral_tokenizer = AutoTokenizer.from_pretrained(model_path)

mistral_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

# ==========================================
# 2. DATA PROCESSING & INFERENCE (UNCHANGED)
# ==========================================

def load_song_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        return json.load(file)

def generate_sarcasm(song_data):
    
    track = song_data.get("track_name", "Unknown Track")
    artist = song_data.get("artist_name", "Unknown Artist")
    lyrics = song_data.get("plain_lyrics", "")
    
    truncated_lyrics = lyrics[:1000] if len(lyrics) > 1000 else lyrics

    prompt = f"""<s>[INST] You are a highly judgmental, sarcastic, witty, dark humor friend reacting to someone's music taste in a group chat.

### SONG
Track: "{track}" by {artist}

### LYRICS
{truncated_lyrics}

### TASK
React directly to the user as if you just heard this playing on their phone.

CRITICAL RULES:
1. MUST reference or twist at least one lyric.
2. MUST reuse 2–5 exact words from the lyrics.
3. Keep it conversational, informal, personal, and highly aggressive.
4. Use biting insults, but avoid generic words like "garbage" or "trash".
5. DO NOT use dialogue format.
6. NO repetition patterns.
7. EXACTLY 1–2 sentences.

Start naturally and STOP after your second sentence. [/INST]"""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = mistral_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = mistral_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.2,  # 🔥 reduced from 2 (too aggressive before)
            do_sample=True,
            pad_token_id=mistral_tokenizer.eos_token_id
        )

    full_response = mistral_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_response.split("[/INST]")[-1].strip()

# ==========================================
# 3. EXECUTION
# ==========================================

if __name__ == "__main__":
    try:
        filepath = "song_data.json"
        
        full_json = load_song_data(filepath)
        song_list = full_json.get("data", [])
        
        if not song_list:
            print("No songs found.")
        else:
            print(f"Found {len(song_list)} songs. Processing all...\n")
            
            for song in song_list:
                track_name = song.get('track_name', 'Unknown Track')
                artist_name = song.get('artist_name', 'Unknown Artist')
                
                print(f"--- Processing: {track_name} by {artist_name} ---")
                
                roast = generate_sarcasm(song)
                print(f"Roast: {roast}\n")
                
    except FileNotFoundError:
        print(f"Error: Could not find the file at {filepath}.")
    except Exception as e:
        print(f"Error: {e}")

c:\Users\samar\OneDrive\Desktop\prodSt\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Mistral Model from local dataset...


c:\Users\samar\OneDrive\Desktop\prodSt\.venv\Lib\site-packages\transformers\quantizers\auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 